# HearAssist — Transcription de la darija avec Qwen3-ASR

Qwen3-ASR est le modèle de transcription de Qwen (gère l'arabe, 52 langues et dialectes).
On suit la même démarche que pour Whisper :

1. **Préparation** : GPU, installation, code et token Hugging Face
2. **Exploration rapide** du dataset `adiren7/darija_speech_to_text`
3. **Référence sans entraînement** : Qwen3-ASR 0.6B et 1.7B
4. **Test rapide** du fine-tuning LoRA (quelques minutes)
5. **Fine-tuning réel** (à lancer avec *Save Version → Save & Run All*)
6. **Évaluation finale** et tableau comparatif

**Réglages Kaggle (panneau de droite) :** Accelerator = **GPU T4 x2** · Internet = **On** ·
Persistence = **Files only** · Input = le dataset Kaggle contenant le dossier `darija_asr`.

**À savoir :** le T4 ne gère pas le `bfloat16` (format natif de Qwen). Le notebook utilise le `float16`.
Si la loss devient `nan` ou si les transcriptions sont vides/absurdes, ajoute `--fp32` aux commandes.

## 1. Préparation

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # un seul GPU : plus simple et plus stable
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# transformers >= 5.13 est obligatoire pour Qwen3-ASR
!pip install -q "transformers>=5.13" "datasets>=3.0" "peft>=0.13" accelerate jiwer soundfile scipy
# torchao préinstallé sur Kaggle (trop ancien) fait planter LoRA (peft) ; on n'en a pas besoin
!pip uninstall -y -q torchao
import transformers; print("transformers", transformers.__version__)

In [ ]:
# Copie du code (dataset Kaggle contenant darija_asr/) vers le dossier de travail
import glob, shutil
src = glob.glob("/kaggle/input/**/finetune_qwen_asr.py", recursive=True)
assert src, "Ajoute la dernière version du dataset Kaggle contenant darija_asr (avec finetune_qwen_asr.py)"
CODE = "/kaggle/working/darija_asr"
shutil.copytree(os.path.dirname(src[0]), CODE, dirs_exist_ok=True)
os.chdir(CODE)
print("Code prêt dans", CODE, ":", sorted(os.listdir(CODE)))

In [ ]:
# Token Hugging Face (facultatif pour adiren7, nécessaire pour pousser un modèle sur le Hub)
# Ajoute-le dans Add-ons -> Secrets sous le nom HF_TOKEN (et coche la case pour l'attacher)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Connecté à Hugging Face")
except Exception as e:
    print("Pas de token HF (ce n'est pas bloquant pour commencer) :", type(e).__name__)

## 2. Exploration rapide
Téléchargement depuis Hugging Face (~1,6 Go), puis essai de Qwen3-ASR sur un exemple.

In [ ]:
import sys
sys.path.insert(0, CODE)
from data import load_darija, decode_audio, clip_duration
from text_norm import normalize

val = load_darija("validation")
print(val)

In [ ]:
# Premier essai : un clip, sa transcription de référence et celle de Qwen3-ASR-0.6B
import torch
from IPython.display import Audio, display
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3-ASR-0.6B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to("cuda").eval()

ex = val[0]
audio = decode_audio(ex["audio"])
display(Audio(audio, rate=16000))
inputs = processor.apply_transcription_request([audio], language=["Arabic"]).to(model.device, model.dtype)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
hyp = processor.decode(out[:, inputs["input_ids"].shape[1]:], return_format="transcription_only")[0]
print("RÉFÉRENCE :", ex["text"])
print("QWEN      :", hyp)

del model; torch.cuda.empty_cache()   # libère le GPU pour la suite

## 3. Référence sans entraînement
WER et CER sur 200 clips de validation, **avant** tout fine-tuning.
Si une sortie paraît vide ou absurde, relance avec `--fp32`.

In [ ]:
N = 200
for model_id in ["Qwen/Qwen3-ASR-0.6B-hf", "Qwen/Qwen3-ASR-1.7B-hf"]:
    print(f"\n===== qwen : {model_id}")
    !python eval_asr.py --backend qwen --model {model_id} --max-samples {N} --batch-size 8 --out-dir /kaggle/working/results

In [ ]:
import glob, json
import pandas as pd
def results_table():
    res = [json.load(open(f)) for f in glob.glob("/kaggle/working/results/*_metrics.json")]
    return pd.DataFrame(res)[["model", "n_samples", "wer", "cer", "rtf", "seconds_per_clip"]].sort_values("wer")
results_table()

In [ ]:
# Analyse d'erreurs : quelques références et prédictions du modèle 0.6B
preds = pd.read_csv(sorted(glob.glob("/kaggle/working/results/Qwen3-ASR-0.6B-hf_*_predictions.csv"))[0])
for _, r in preds.head(5).iterrows():
    print("REF :", r["ref_norm"][:200]); print("HYP :", r["hyp_norm"][:200]); print()

## 4. Test rapide du fine-tuning
Vérifie en quelques minutes que le fine-tuning LoRA, la sauvegarde et l'évaluation tournent sur le GPU.
Les scores obtenus ici n'ont aucune valeur (10 étapes seulement).

In [ ]:
!python finetune_qwen_asr.py --max-train-samples 32 --eval-samples 8 --max-steps 10 --eval-steps 10 \
    --warmup-steps 0 --batch-size 2 --grad-accum 1 --output-dir /kaggle/working/smoke_qwen
!python eval_asr.py --backend qwen --model /kaggle/working/smoke_qwen/final --max-samples 16 --out-dir /kaggle/working/smoke_results

## 5. Fine-tuning réel
⚠️ Plusieurs heures. Passe `RUN_TRAINING = True`, puis lance **Save Version → Save & Run All**
pour que l'entraînement continue navigateur fermé.

- Commence par **0.6B** ; passe à **1.7B** seulement si le gain de la référence le justifie
  (réduis alors `--batch-size` à 2 et monte `--grad-accum` à 8).
- Le meilleur checkpoint est choisi sur la **loss de validation** ; le WER se mesure à l'étape 6.
- Avec `HUB_REPO` renseigné (token HF requis), les checkpoints sont aussi poussés sur le Hub.
  Après une coupure, ajoute `--resume` pour reprendre.

In [ ]:
RUN_TRAINING = False
BASE = "Qwen/Qwen3-ASR-0.6B-hf"
HUB_REPO = ""          # ex. "ton-pseudo/qwen3-asr-darija"
OUT = "/kaggle/working/qwen3-asr-darija"

if RUN_TRAINING:
    hub = f"--push-to-hub {HUB_REPO}" if HUB_REPO else ""
    !python finetune_qwen_asr.py --base-model {BASE} --output-dir {OUT} \
        --batch-size 4 --grad-accum 4 --max-steps 2000 --eval-steps 200 {hub}

## 6. Évaluation finale

In [ ]:
if os.path.exists(f"{OUT}/final"):
    # 200 clips : même échantillon que la référence (comparaison avant / après)
    !python eval_asr.py --backend qwen --model {OUT}/final --max-samples 200 --batch-size 8 --out-dir /kaggle/working/results
    # Validation complète (1 065 clips)
    !python eval_asr.py --backend qwen --model {OUT}/final --max-samples 0 --batch-size 8 --out-dir /kaggle/working/results
else:
    print("Pas encore de modèle entraîné : lance d'abord l'étape 5.")

In [ ]:
results_table()